# 17 finalize annotation

**Purpose:** Create the final annotation and compartment object.

**Original analysis notebook:** `17_final_annotation_figure_consolidation_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 36 — Final annotation & figure consolidation  
## HVG8000 / scVI r0.6 / marker + CopyKAT consensus

This notebook consolidates the final marker + CopyKAT + infercnvpy-reviewed annotation, creates clean final labels, saves a final `.h5ad`, and exports thesis-ready QC tables and figures.

It does **not** redo scVI, clustering, DE, CopyKAT, or infercnvpy.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import scanpy as sc

warnings.filterwarnings("ignore")

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "integrated"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures" / "final_annotation_CNV_consensus_HVG8000_r0_6"
for d in [PROCESSED_DIR, METADATA_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

INPUT_H5AD = PROCESSED_DIR / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_consensus.h5ad"
FINAL_H5AD = PROCESSED_DIR / "GBM_core_4datasets_final_annotation_CNV_consensus.h5ad"
FINAL_ALIAS_H5AD = PROCESSED_DIR / "GBM_core_4datasets_final_annotation.h5ad"

CLUSTER_KEY = "leiden_scvi_main"
COPYKAT_KEY = "copykat_pred_clean"
FINAL_GROUP_KEY = "final_malignancy_group_copykat_marker"

print("INPUT_H5AD:", INPUT_H5AD)
print("exists:", INPUT_H5AD.exists())


In [ ]:
def write_tsv_replace(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=index)
    print(f"[SAVED] {path} shape={df.shape}")


def write_h5ad_replace(adata_obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    adata_obj.write_h5ad(path)
    print(f"[SAVED] {path}")


def choose_first_existing_column(df, candidates, required=True, label="column"):
    for c in candidates:
        if c in df.columns:
            print(f"[FOUND] {label}: {c}")
            return c
    if required:
        raise KeyError(f"None of these {label} candidates found: {candidates}")
    print(f"[MISSING] {label}: none of {candidates}")
    return None


def clean_cluster_label(x):
    x = str(x)
    try:
        return str(int(float(x)))
    except Exception:
        return x


def save_current_fig(stem, dpi=180):
    png = FIGURES_DIR / f"{stem}.png"
    pdf = FIGURES_DIR / f"{stem}.pdf"
    plt.savefig(png, dpi=dpi, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    plt.close()
    print(f"[FIGURE] {png}")
    return str(png), str(pdf)


In [ ]:
if not INPUT_H5AD.exists():
    raise FileNotFoundError(f"Missing input file: {INPUT_H5AD}")

adata = sc.read_h5ad(INPUT_H5AD)
print(adata)
print("obs columns:", len(adata.obs.columns))
print("var columns:", len(adata.var.columns))
print("obsm keys:", list(adata.obsm.keys()))

required_cols = [CLUSTER_KEY, COPYKAT_KEY, FINAL_GROUP_KEY]
missing = [c for c in required_cols if c not in adata.obs.columns]
if missing:
    raise KeyError(f"Missing required columns in adata.obs: {missing}")
if "X_umap" not in adata.obsm:
    raise KeyError("X_umap not found in adata.obsm.")

CONDITION_KEY = choose_first_existing_column(adata.obs, ["condition", "disease_condition", "diagnosis", "group"], False, "condition")
DATASET_KEY = choose_first_existing_column(adata.obs, ["core_dataset", "dataset_id", "dataset", "study"], False, "dataset")
SAMPLE_KEY = choose_first_existing_column(adata.obs, ["sample_id", "sample", "orig.ident", "sample_name"], False, "sample")
PATIENT_KEY = choose_first_existing_column(adata.obs, ["patient_id", "patient", "donor_id", "case_id"], False, "patient")
MARKER_ANNOT_KEY = choose_first_existing_column(adata.obs, ["annotation_marker_based", "marker_based_annotation", "cluster_marker_annotation", "annotation", "cell_type_marker_based"], False, "marker annotation")

adata.obs[CLUSTER_KEY] = adata.obs[CLUSTER_KEY].astype(str).map(clean_cluster_label).astype("category")
adata.obs[COPYKAT_KEY] = adata.obs[COPYKAT_KEY].astype(str).astype("category")
adata.obs[FINAL_GROUP_KEY] = adata.obs[FINAL_GROUP_KEY].astype(str).astype("category")


In [ ]:
cluster_to_final_annotation = {
    "0":  "malignant_OPC_NPC_like_CNV_supported",
    "1":  "review_myeloid_C1Q_TREM2_APOE_TAM_mixed_CNV",
    "2":  "cycling_malignant_like_candidate_CNV_supported",
    "3":  "review_myeloid_inflammatory_MES_associated_mixed_CNV",
    "4":  "malignant_hypoxia_stress_CNV_supported",
    "5":  "review_T_NK_cells_dataset_biased_mixed_CNV",
    "6":  "review_myeloid_APC_HLA_CD163_TAM_mixed_CNV",
    "7":  "malignant_AC_like_reactive_CNV_supported",
    "8":  "non_malignant_oligodendrocyte_myelinating_reference_like",
    "9":  "review_ambiguous_mixed_Hb_T_NK_HLA",
    "10": "review_myeloid_FCN1_inflammatory_monocyte_CNV_discordant",
    "11": "malignant_NPC_neuronal_like_CNV_supported",
    "12": "review_MES_ECM_stromal_like_ambiguous_CNV_high",
    "13": "non_malignant_mural_pericyte_fibroblast_like",
    "14": "non_malignant_endothelial_reference_like",
    "15": "non_malignant_B_plasma_like",
    "16": "non_malignant_rare_mast_cell_like_low_count",
    "17": "review_erythroid_Hb_high_or_ambient_RBC",
}

cluster_to_major_compartment = {
    "0": "malignant_like", "4": "malignant_like", "7": "malignant_like", "11": "malignant_like",
    "2": "cycling_malignant_like_candidate",
    "8": "non_malignant_or_diploid_leaning", "13": "non_malignant_or_diploid_leaning", "14": "non_malignant_or_diploid_leaning", "15": "non_malignant_or_diploid_leaning", "16": "non_malignant_or_diploid_leaning",
    "1": "review", "3": "review", "5": "review", "6": "review", "9": "review", "10": "review", "12": "review", "17": "review",
}

cluster_to_analysis_use = {
    "0": "use_malignant", "4": "use_malignant", "7": "use_malignant", "11": "use_malignant",
    "2": "use_cycling_candidate_separately",
    "8": "use_reference_non_malignant", "14": "use_reference_non_malignant",
    "13": "use_non_malignant_with_caution", "15": "use_non_malignant_with_caution", "16": "use_non_malignant_with_caution_low_count",
    "1": "review_exclude_from_core_malignant", "3": "review_exclude_from_core_malignant", "5": "review_exclude_from_core_malignant", "6": "review_exclude_from_core_malignant",
    "9": "review_exclude_from_core_malignant", "10": "review_exclude_from_core_malignant_CNV_discordant", "12": "review_exclude_from_core_malignant_ambiguous_CNV_high", "17": "review_exclude_low_count_Hb_high",
}

adata.obs["final_annotation_refined"] = adata.obs[CLUSTER_KEY].astype(str).map(cluster_to_final_annotation).fillna("unmapped_cluster_review").astype("category")
adata.obs["final_compartment_major"] = adata.obs[CLUSTER_KEY].astype(str).map(cluster_to_major_compartment).fillna("review").astype("category")
adata.obs["final_analysis_use"] = adata.obs[CLUSTER_KEY].astype(str).map(cluster_to_analysis_use).fillna("review").astype("category")

simple_map = {
    "malignant_like": "malignant_like",
    "cycling_malignant_like_candidate": "cycling_candidate",
    "non_malignant_or_diploid_leaning": "non_malignant_diploid_leaning",
    "review": "review_ambiguous_discordant",
}
adata.obs["final_compartment_simple"] = adata.obs["final_compartment_major"].astype(str).map(simple_map).astype("category")

adata.obs[["final_annotation_refined", "final_compartment_major", "final_analysis_use", "final_compartment_simple"]].head()


In [ ]:
readiness = pd.DataFrame([
    {"metric": "input_h5ad", "value": str(INPUT_H5AD)},
    {"metric": "n_cells", "value": int(adata.n_obs)},
    {"metric": "n_genes", "value": int(adata.n_vars)},
    {"metric": "cluster_key", "value": CLUSTER_KEY},
    {"metric": "copykat_key", "value": COPYKAT_KEY},
    {"metric": "final_group_key_previous", "value": FINAL_GROUP_KEY},
    {"metric": "has_X_umap", "value": "X_umap" in adata.obsm},
    {"metric": "has_X_scVI", "value": "X_scVI" in adata.obsm},
    {"metric": "final_annotation_refined_created", "value": "final_annotation_refined" in adata.obs.columns},
])
READINESS_TSV = METADATA_DIR / "final_annotation_consolidation_readiness_HVG8000_r0_6.tsv"
write_tsv_replace(readiness, READINESS_TSV)

cell_counts = []
for key in [CLUSTER_KEY, "final_annotation_refined", "final_compartment_major", "final_compartment_simple", "final_analysis_use", COPYKAT_KEY]:
    vc = adata.obs[key].astype(str).value_counts(dropna=False).rename_axis("label").reset_index(name="n_cells")
    vc["fraction"] = vc["n_cells"] / adata.n_obs
    vc["category"] = key
    cell_counts.append(vc)
cell_counts_df = pd.concat(cell_counts, ignore_index=True)
CELL_COUNTS_TSV = METADATA_DIR / "final_annotation_consolidated_cell_counts_HVG8000_r0_6.tsv"
write_tsv_replace(cell_counts_df, CELL_COUNTS_TSV)
display(cell_counts_df.head(20))


In [ ]:
obs = adata.obs.copy()
obs["_cluster"] = obs[CLUSTER_KEY].astype(str)

rows = []
for cl, sub in obs.groupby("_cluster", observed=True):
    row = {
        "cluster": cl,
        "n_cells": int(sub.shape[0]),
        "fraction_total": float(sub.shape[0] / obs.shape[0]),
        "final_annotation_refined": sub["final_annotation_refined"].astype(str).mode().iloc[0],
        "final_compartment_major": sub["final_compartment_major"].astype(str).mode().iloc[0],
        "final_analysis_use": sub["final_analysis_use"].astype(str).mode().iloc[0],
    }
    if MARKER_ANNOT_KEY is not None:
        row["original_marker_annotation_mode"] = sub[MARKER_ANNOT_KEY].astype(str).mode().iloc[0]
    copy_counts = sub[COPYKAT_KEY].astype(str).value_counts()
    defined = int(copy_counts.get("aneuploid", 0) + copy_counts.get("diploid", 0))
    row["copykat_aneuploid_n"] = int(copy_counts.get("aneuploid", 0))
    row["copykat_diploid_n"] = int(copy_counts.get("diploid", 0))
    row["copykat_not_defined_n"] = int(copy_counts.get("not.defined", 0) + copy_counts.get("not_defined", 0))
    row["copykat_defined_n"] = defined
    row["copykat_aneuploid_fraction_defined"] = float(row["copykat_aneuploid_n"] / defined) if defined > 0 else np.nan
    if CONDITION_KEY:
        row["top_condition"] = sub[CONDITION_KEY].astype(str).value_counts().idxmax()
        row["top_condition_fraction"] = float(sub[CONDITION_KEY].astype(str).value_counts(normalize=True).max())
    if DATASET_KEY:
        row["top_dataset"] = sub[DATASET_KEY].astype(str).value_counts().idxmax()
        row["top_dataset_fraction"] = float(sub[DATASET_KEY].astype(str).value_counts(normalize=True).max())
    if SAMPLE_KEY:
        row["top_sample"] = sub[SAMPLE_KEY].astype(str).value_counts().idxmax()
        row["top_sample_fraction"] = float(sub[SAMPLE_KEY].astype(str).value_counts(normalize=True).max())
    if PATIENT_KEY:
        row["top_patient"] = sub[PATIENT_KEY].astype(str).value_counts().idxmax()
        row["top_patient_fraction"] = float(sub[PATIENT_KEY].astype(str).value_counts(normalize=True).max())
    rows.append(row)

cluster_summary = pd.DataFrame(rows)
cluster_summary["cluster_numeric"] = pd.to_numeric(cluster_summary["cluster"], errors="coerce")
cluster_summary = cluster_summary.sort_values(["cluster_numeric", "cluster"]).drop(columns=["cluster_numeric"])
CLUSTER_SUMMARY_TSV = METADATA_DIR / "final_annotation_consolidated_cluster_summary_HVG8000_r0_6.tsv"
write_tsv_replace(cluster_summary, CLUSTER_SUMMARY_TSV)
display(cluster_summary)


In [ ]:
composition_paths = {}

if CONDITION_KEY:
    cond_counts = obs.groupby([CONDITION_KEY, "final_compartment_major"], observed=True).size().reset_index(name="n_cells")
    cond_counts["condition_total_cells"] = cond_counts.groupby(CONDITION_KEY)["n_cells"].transform("sum")
    cond_counts["fraction_within_condition"] = cond_counts["n_cells"] / cond_counts["condition_total_cells"]
    path = METADATA_DIR / "final_annotation_condition_compartment_counts_HVG8000_r0_6.tsv"
    write_tsv_replace(cond_counts, path)
    composition_paths["condition_compartment_counts"] = str(path)

if DATASET_KEY:
    dataset_counts = obs.groupby([DATASET_KEY, "final_compartment_major"], observed=True).size().reset_index(name="n_cells")
    dataset_counts["dataset_total_cells"] = dataset_counts.groupby(DATASET_KEY)["n_cells"].transform("sum")
    dataset_counts["fraction_within_dataset"] = dataset_counts["n_cells"] / dataset_counts["dataset_total_cells"]
    path = METADATA_DIR / "final_annotation_dataset_compartment_counts_HVG8000_r0_6.tsv"
    write_tsv_replace(dataset_counts, path)
    composition_paths["dataset_compartment_counts"] = str(path)

if SAMPLE_KEY:
    group_cols = [SAMPLE_KEY]
    for k in [PATIENT_KEY, CONDITION_KEY, DATASET_KEY]:
        if k and k not in group_cols:
            group_cols.append(k)
    group_cols.append("final_compartment_major")
    sample_counts = obs.groupby(group_cols, observed=True).size().reset_index(name="n_cells")
    sample_counts["sample_total_cells"] = sample_counts.groupby(SAMPLE_KEY)["n_cells"].transform("sum")
    sample_counts["fraction_within_sample"] = sample_counts["n_cells"] / sample_counts["sample_total_cells"]
    path = METADATA_DIR / "final_annotation_sample_level_compartment_composition_HVG8000_r0_6.tsv"
    write_tsv_replace(sample_counts, path)
    composition_paths["sample_level_compartment_composition"] = str(path)

print(json.dumps(composition_paths, indent=2))


In [ ]:
marker_panel = {
    "malignant_stem_like": ["SOX2", "NES", "PROM1", "CD44", "VIM"],
    "OPC_NPC_like": ["PTPRZ1", "BCAN", "OLIG1", "OLIG2", "PDGFRA", "SOX4", "DLL3"],
    "AC_like_reactive": ["GFAP", "AQP4", "ALDOC", "CLU", "SPARCL1", "SLC1A3"],
    "hypoxia_stress": ["CA9", "VEGFA", "HILPDA", "LOX", "ADM", "NDRG1"],
    "cycling": ["MKI67", "TOP2A", "UBE2C", "BIRC5", "PBK", "CENPF"],
    "myeloid_TAM": ["C1QA", "C1QB", "C1QC", "TREM2", "APOE", "CSF1R", "AIF1", "TYROBP", "CD68"],
    "monocyte_inflammatory": ["FCN1", "S100A8", "S100A9", "LYZ", "LST1", "AQP9"],
    "T_NK": ["CD3D", "CD3E", "TRAC", "NKG7", "GZMA", "GNLY", "LCK"],
    "oligodendrocyte": ["MOG", "MAG", "MOBP", "PLP1", "ERMN"],
    "endothelial": ["PECAM1", "VWF", "KDR", "ERG", "ESAM", "EMCN", "CLEC14A"],
    "mural_pericyte": ["PDGFRB", "RGS5", "ACTA2", "MCAM", "ENPEP", "CD248"],
    "B_plasma": ["CD79A", "CD79B", "MS4A1", "MZB1", "JCHAIN", "IGHM"],
    "erythroid_Hb": ["HBA1", "HBA2", "HBB", "ALAS2"],
}
rows = []
varnames = set(map(str, adata.var_names))
for panel, genes in marker_panel.items():
    for g in genes:
        rows.append({"panel": panel, "gene": g, "present": g in varnames})
marker_presence = pd.DataFrame(rows)
MARKER_PRESENCE_TSV = METADATA_DIR / "final_annotation_marker_panel_presence_HVG8000_r0_6.tsv"
write_tsv_replace(marker_presence, MARKER_PRESENCE_TSV)
display(marker_presence.groupby("panel")["present"].agg(["sum", "count"]))


In [ ]:
sc.settings.figdir = str(FIGURES_DIR)
sc.set_figure_params(figsize=(7, 6), dpi=120, frameon=False)
figure_records = []

umap_keys = [
    ("final_compartment_major", "UMAP_final_compartment_major"),
    ("final_annotation_refined", "UMAP_final_annotation_refined"),
    ("final_analysis_use", "UMAP_final_analysis_use"),
    (COPYKAT_KEY, "UMAP_CopyKAT_call"),
    (CLUSTER_KEY, "UMAP_leiden_scvi_main"),
]
if CONDITION_KEY:
    umap_keys.append((CONDITION_KEY, "UMAP_condition"))
if DATASET_KEY:
    umap_keys.append((DATASET_KEY, "UMAP_dataset"))

for key, stem in umap_keys:
    print("[PLOT]", key)
    sc.pl.umap(adata, color=key, legend_loc="right margin", size=2, show=False, title=key)
    png, pdf = save_current_fig(f"final_annotation_{stem}_HVG8000_r0_6")
    figure_records.append({"figure": stem, "type": "umap", "color_key": key, "png": png, "pdf": pdf})

FIGURE_MANIFEST_TSV = METADATA_DIR / "final_annotation_consolidation_figure_manifest_HVG8000_r0_6.tsv"
figure_manifest = pd.DataFrame(figure_records)
write_tsv_replace(figure_manifest, FIGURE_MANIFEST_TSV)


In [ ]:
# Bar plot: final compartments
plot_df = obs["final_compartment_major"].astype(str).value_counts().rename_axis("final_compartment_major").reset_index(name="n_cells")
plt.figure(figsize=(7, 4))
plt.bar(plot_df["final_compartment_major"], plot_df["n_cells"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of cells")
plt.title("Final compartment counts")
png, pdf = save_current_fig("final_annotation_bar_final_compartment_counts_HVG8000_r0_6")
figure_manifest = pd.concat([figure_manifest, pd.DataFrame([{"figure": "bar_final_compartment_counts", "type": "bar", "png": png, "pdf": pdf}])], ignore_index=True)

# Stacked bar: CopyKAT by final compartment
ct = pd.crosstab(obs["final_compartment_major"].astype(str), obs[COPYKAT_KEY].astype(str))
prop = ct.div(ct.sum(axis=1), axis=0).fillna(0)
plt.figure(figsize=(7, 4))
bottom = np.zeros(prop.shape[0])
x = np.arange(prop.shape[0])
for col in prop.columns:
    plt.bar(x, prop[col].values, bottom=bottom, label=col)
    bottom += prop[col].values
plt.xticks(x, prop.index, rotation=45, ha="right")
plt.ylabel("Fraction")
plt.title("CopyKAT composition by final compartment")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
png, pdf = save_current_fig("final_annotation_bar_copykat_by_final_compartment_HVG8000_r0_6")
figure_manifest = pd.concat([figure_manifest, pd.DataFrame([{"figure": "bar_copykat_by_final_compartment", "type": "stacked_bar", "png": png, "pdf": pdf}])], ignore_index=True)
write_tsv_replace(figure_manifest, FIGURE_MANIFEST_TSV)


In [ ]:
# Marker dotplot by refined annotation. This is lightweight because it subsets to marker genes only.
ordered_panels = list(marker_panel.keys())
marker_genes_present = []
for panel in ordered_panels:
    for g in marker_panel[panel]:
        if g in adata.var_names and g not in marker_genes_present:
            marker_genes_present.append(g)

print("n marker genes present:", len(marker_genes_present))

if len(marker_genes_present) > 0:
    adata_plot = adata[:, marker_genes_present].copy()
    if "counts" in adata_plot.layers:
        adata_plot.X = adata_plot.layers["counts"].copy()
    elif "raw_counts" in adata_plot.layers:
        adata_plot.X = adata_plot.layers["raw_counts"].copy()
    sc.pp.normalize_total(adata_plot, target_sum=1e4)
    sc.pp.log1p(adata_plot)
    sc.pl.dotplot(
        adata_plot,
        var_names=marker_genes_present,
        groupby="final_annotation_refined",
        standard_scale="var",
        dendrogram=False,
        show=False,
    )
    png, pdf = save_current_fig("final_annotation_marker_dotplot_by_refined_annotation_HVG8000_r0_6", dpi=180)
    figure_manifest = pd.concat([figure_manifest, pd.DataFrame([{"figure": "marker_dotplot_by_refined_annotation", "type": "dotplot", "png": png, "pdf": pdf}])], ignore_index=True)
    write_tsv_replace(figure_manifest, FIGURE_MANIFEST_TSV)
else:
    print("[WARNING] No marker genes present for dotplot.")


In [ ]:
adata.uns["final_annotation_consolidation"] = {
    "input_h5ad": str(INPUT_H5AD),
    "cluster_key": CLUSTER_KEY,
    "copykat_key": COPYKAT_KEY,
    "previous_final_group_key": FINAL_GROUP_KEY,
    "new_columns": ["final_annotation_refined", "final_compartment_major", "final_compartment_simple", "final_analysis_use"],
    "main_malignant_like_clusters": ["0", "4", "7", "11"],
    "cycling_malignant_like_candidate_clusters": ["2"],
    "reference_like_clusters": ["8", "14"],
    "non_malignant_diploid_leaning_clusters": ["8", "13", "14", "15", "16"],
    "review_clusters": ["1", "3", "5", "6", "9", "10", "12", "17"],
}
write_h5ad_replace(adata, FINAL_H5AD)
write_h5ad_replace(adata, FINAL_ALIAS_H5AD)


In [ ]:
manifest_items = [
    {"artifact": "readiness", "path": str(READINESS_TSV), "exists": READINESS_TSV.exists()},
    {"artifact": "cell_counts", "path": str(CELL_COUNTS_TSV), "exists": CELL_COUNTS_TSV.exists()},
    {"artifact": "cluster_summary", "path": str(CLUSTER_SUMMARY_TSV), "exists": CLUSTER_SUMMARY_TSV.exists()},
    {"artifact": "marker_presence", "path": str(MARKER_PRESENCE_TSV), "exists": MARKER_PRESENCE_TSV.exists()},
    {"artifact": "figure_manifest", "path": str(FIGURE_MANIFEST_TSV), "exists": FIGURE_MANIFEST_TSV.exists()},
    {"artifact": "final_h5ad", "path": str(FINAL_H5AD), "exists": FINAL_H5AD.exists()},
    {"artifact": "final_alias_h5ad", "path": str(FINAL_ALIAS_H5AD), "exists": FINAL_ALIAS_H5AD.exists()},
]
for k, p in composition_paths.items():
    manifest_items.append({"artifact": k, "path": p, "exists": Path(p).exists()})

OUTPUT_MANIFEST_TSV = METADATA_DIR / "final_annotation_consolidation_output_manifest_HVG8000_r0_6.tsv"
output_manifest = pd.DataFrame(manifest_items)
write_tsv_replace(output_manifest, OUTPUT_MANIFEST_TSV)
display(output_manifest)
print("[DONE] Final annotation consolidation completed.")


## After running this notebook

Please send these TSV files:

```text
${GLIOMA_PROJECT_ROOT}/metadata/final_annotation_consolidation_readiness_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/final_annotation_consolidated_cell_counts_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/final_annotation_consolidated_cluster_summary_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/final_annotation_condition_compartment_counts_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/final_annotation_sample_level_compartment_composition_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/final_annotation_marker_panel_presence_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/final_annotation_consolidation_figure_manifest_HVG8000_r0_6.tsv
${GLIOMA_PROJECT_ROOT}/metadata/final_annotation_consolidation_output_manifest_HVG8000_r0_6.tsv
```

Do **not** upload the `.h5ad`; it will be large.